# SupportIQ — DistilBERT Fine-Tuning
**Team 12 · University of Houston**

Fine-tunes `distilbert-base-uncased` on Jira Issues to classify:
**Bug · New Feature · Improvement · Task · Test**

### Before running:
1. `Runtime → Change runtime type → T4 GPU`
2. Upload `jira_issues_50k.csv` when prompted
3. Run all cells (~15 minutes total)

In [1]:
# Cell 1 — Install
!pip install transformers datasets torch scikit-learn pandas tqdm -q

In [2]:
# Cell 2 — Upload CSV
from google.colab import files
print('Upload your jira_issues_50k.csv:')
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]
print(f'Uploaded: {csv_path}')

Upload your jira_issues_50k.csv:


Saving jira_issues_50k.csv to jira_issues_50k.csv
Uploaded: jira_issues_50k.csv


In [3]:
# Cell 3 — Load, clean, consolidate issue types
import re, json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

ISSUE_TYPE_MAP = {
    'bug':'Bug','defect':'Bug','error':'Bug','regression':'Bug','patch':'Bug',
    'new feature':'New Feature','feature request':'New Feature',
    'feature':'New Feature','wish':'New Feature',
    'improvement':'Improvement','enhancement':'Improvement',
    'refactoring':'Improvement','optimisation':'Improvement','optimization':'Improvement',
    'task':'Task','sub-task':'Task','story':'Task','epic':'Task',
    'documentation':'Task','component upgrade':'Task','dependency upgrade':'Task',
    'test':'Test','testing':'Test',
}

def clean_text(x):
    if pd.isna(x): return ''
    x = str(x)
    x = re.sub(r'http\S+', ' ', x)
    x = re.sub(r'<.*?>', ' ', x)
    x = re.sub(r'\{code[^}]*\}.*?\{code\}', ' ', x, flags=re.S)
    x = re.sub(r'\s+', ' ', x).strip()
    return x[:512]   # truncate early — DistilBERT max is 512 tokens anyway

def find_col(cols, candidates):
    lmap = {str(c).lower().strip(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lmap: return lmap[cand.lower()]
    return None

df_raw = pd.read_csv(csv_path)
print(f'Raw: {len(df_raw):,} rows | Cols: {df_raw.columns.tolist()}')

title_col = find_col(df_raw.columns, ['title','summary','subject'])
desc_col  = find_col(df_raw.columns, ['description','body','details','text'])
type_col  = find_col(df_raw.columns, ['issue_type','issuetype','type'])
print(f'Title: {title_col} | Desc: {desc_col} | Type: {type_col}')

df = pd.DataFrame()
df['text']  = (df_raw[title_col].apply(clean_text) + ' ' +
               df_raw[desc_col].apply(clean_text)).str.strip()
df['label'] = df_raw[type_col].apply(lambda x: ISSUE_TYPE_MAP.get(str(x).lower().strip()))
df = df[df['label'].notna() & (df['text'].str.len() >= 20)].drop_duplicates('text').reset_index(drop=True)

print(f'\nClean rows: {len(df):,}')
print(df['label'].value_counts())

Raw: 50,000 rows | Cols: ['id', 'created', 'description', 'key', 'priority', 'project', 'project_name', 'repositoryname', 'resolution', 'resolved', 'status', 'title', 'type', 'updated', 'votes', 'watchers', 'assignee_id', 'reporter_id']
Title: title | Desc: description | Type: type

Clean rows: 49,404
label
Bug            27014
Improvement     9416
Task            7252
New Feature     5482
Test             240
Name: count, dtype: int64


In [4]:
# Cell 4 — Cap per class at 8000 samples (keeps dataset manageable, ~15 min training)
# Minority classes get oversampled up to median, majority classes get capped
CAP_PER_CLASS = 8000
MIN_PER_CLASS = 500

balanced = []
for label, group in df.groupby('label'):
    if len(group) < MIN_PER_CLASS:
        # oversample
        group = group.sample(MIN_PER_CLASS, replace=True, random_state=42)
    elif len(group) > CAP_PER_CLASS:
        # downsample majority
        group = group.sample(CAP_PER_CLASS, random_state=42)
    balanced.append(group)

df_bal = pd.concat(balanced).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Balanced dataset: {len(df_bal):,} rows')
print(df_bal['label'].value_counts())

le = LabelEncoder()
df_bal['label_id'] = le.fit_transform(df_bal['label'])
LABELS     = list(le.classes_)
NUM_LABELS = len(LABELS)
print(f'\nLabels: {dict(enumerate(LABELS))}')

Balanced dataset: 29,234 rows
label
Improvement    8000
Bug            8000
Task           7252
New Feature    5482
Test            500
Name: count, dtype: int64

Labels: {0: 'Bug', 1: 'Improvement', 2: 'New Feature', 3: 'Task', 4: 'Test'}


In [5]:
# Cell 5 — Tokenize
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader

print('GPU:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_tr, X_te, y_tr, y_te = train_test_split(
    df_bal['text'].tolist(), df_bal['label_id'].tolist(),
    test_size=0.15, random_state=42, stratify=df_bal['label_id']
)
print(f'Train: {len(X_tr):,} | Val: {len(X_te):,}')

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class JiraDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc    = tokenizer(texts, truncation=True, padding=True,
                                max_length=128, return_tensors='pt')
        self.labels = torch.tensor(labels)
    def __len__(self):  return len(self.labels)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

train_ds = JiraDataset(X_tr, y_tr)
val_ds   = JiraDataset(X_te, y_te)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f'Batches per epoch: {len(train_dl)}')

GPU: True
Train: 24,848 | Val: 4,386


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Batches per epoch: 777


In [6]:
# Cell 6 — Fine-tune with progress bar
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.notebook import tqdm

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
).to(device)

EPOCHS    = 3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps//10,
    num_training_steps=total_steps
)

best_f1 = 0
for epoch in range(EPOCHS):
    print(f'\n── Epoch {epoch+1}/{EPOCHS} ──')

    # Train
    model.train()
    total_loss = 0
    pbar = tqdm(train_dl, desc='Training', leave=True)
    for batch_inputs, batch_labels in pbar:
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        batch_labels = batch_labels.to(device)
        optimizer.zero_grad()
        out  = model(**batch_inputs, labels=batch_labels)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(train_dl)

    # Evaluate
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch_inputs, batch_labels in tqdm(val_dl, desc='Evaluating', leave=False):
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            preds = model(**batch_inputs).logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(batch_labels.numpy())

    acc   = accuracy_score(all_true, all_preds)
    macro = f1_score(all_true, all_preds, average='macro', zero_division=0)
    print(f'Loss: {avg_loss:.4f} | Accuracy: {acc:.4f} | Macro F1: {macro:.4f}')

    if macro > best_f1:
        best_f1 = macro
        model.save_pretrained('/content/distilbert_classifier')
        tokenizer.save_pretrained('/content/distilbert_classifier')
        print(f'  ✓ Best model saved (F1={best_f1:.4f})')

print('\n── Final classification report ──')
print(classification_report(all_true, all_preds, target_names=LABELS, zero_division=0))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



── Epoch 1/3 ──


...

Training:   0%|          | 0/777 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/69 [00:00<?, ?it/s]

Loss: 0.7809 | Accuracy: 0.6172 | Macro F1: 0.5925


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (F1=0.5925)

── Final classification report ──
              precision    recall  f1-score   support

         Bug       0.76      0.79      0.78      1200
 Improvement       0.55      0.52      0.53      1200
 New Feature       0.50      0.50      0.50       823
        Task       0.62      0.63      0.62      1088
        Test       0.51      0.56      0.53        75

    accuracy                           0.62      4386
   macro avg       0.59      0.60      0.59      4386
weighted avg       0.61      0.62      0.62      4386



In [7]:
# Cell 7 — Save label map
label_map = {str(i): label for i, label in enumerate(LABELS)}
with open('/content/distilbert_classifier/label_map.json', 'w') as f:
    json.dump(label_map, f)
model.config.id2label  = label_map
model.config.label2id  = {v: k for k, v in label_map.items()}
model.config.save_pretrained('/content/distilbert_classifier')

print('Label map:', label_map)
import os
for f in os.listdir('/content/distilbert_classifier'):
    size = os.path.getsize(f'/content/distilbert_classifier/{f}')
    print(f'  {f}: {size/1024/1024:.1f} MB')

Label map: {'0': 'Bug', '1': 'Improvement', '2': 'New Feature', '3': 'Task', '4': 'Test'}
  tokenizer.json: 0.7 MB
  label_map.json: 0.0 MB
  config.json: 0.0 MB
  model.safetensors: 255.4 MB
  tokenizer_config.json: 0.0 MB


In [8]:
# Cell 8 — Smoke test
from transformers import pipeline as hf_pipeline
clf = hf_pipeline('text-classification', model='/content/distilbert_classifier',
                  tokenizer='/content/distilbert_classifier',
                  device=0 if torch.cuda.is_available() else -1,
                  return_all_scores=False)

tests = [
    ('NullPointerException in JdbcTemplate when ResultSet is empty',                'Bug'),
    ('Add support for virtual threads in Spring WebFlux',                           'New Feature'),
    ('Improve error message when Spring bean dependency is missing',                'Improvement'),
    ('Upgrade Jackson dependency to fix CVE',                                       'Task'),
    ('Missing unit tests for edge cases in Apache Commons Math',                    'Test'),
]
print(f'{"Predicted":15s} {"Expected":15s} {"Text"}')
print('-'*70)
correct = 0
for text, expected in tests:
    result    = clf(text, truncation=True, max_length=128)[0]
    predicted = result['label']
    ok        = '✓' if predicted == expected else '✗'
    if predicted == expected: correct += 1
    print(f'{ok} {predicted:15s} {expected:15s} {text[:50]}')
print(f'\nSmoke test: {correct}/{len(tests)} correct')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Predicted       Expected        Text
----------------------------------------------------------------------
✓ Bug             Bug             NullPointerException in JdbcTemplate when ResultSe
✓ New Feature     New Feature     Add support for virtual threads in Spring WebFlux
✓ Improvement     Improvement     Improve error message when Spring bean dependency 
✓ Task            Task            Upgrade Jackson dependency to fix CVE
✗ Bug             Test            Missing unit tests for edge cases in Apache Common

Smoke test: 4/5 correct


In [9]:
# Cell 9 — Zip and download
!cd /content && zip -r distilbert_classifier.zip distilbert_classifier/ -q
from google.colab import files
files.download('/content/distilbert_classifier.zip')
print('Download started!')
print('Unzip and place distilbert_classifier/ inside your backend/ folder.')
print('Restart the backend — it will auto-load DistilBERT.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started!
Unzip and place distilbert_classifier/ inside your backend/ folder.
Restart the backend — it will auto-load DistilBERT.
